# Direct-publisher smoke intake: CPU-only review draft
Checks mounted wheel bytes against the trusted inventory. Does not install or run a model, establish permission, verify provider version, or authorize a smoke test. Requested dataset version: 1. The attachment version needs separate provider evidence.

In [ ]:
import base64, hashlib, json, pathlib, shutil, sys, tempfile
# CPU integrity only: no pip, package imports, server, GPU queries or launch.
payload = {'__init__.py': 'IiIiQ1BVLW9ubHkgaW50YWtlIGZvciBhIHByb3Bvc2VkIGRpcmVjdC1wdWJsaXNoZXIgc21va2UgdGVzdDsgbm8gbGl2ZSBydW5uZXIuIiIiCg==', 'preflight.py': 'IiIiVmVyaWZ5IG1vdW50ZWQgYnl0ZXMgd2l0aG91dCBpbnN0YWxsaW5nIHBhY2thZ2VzLCBpbXBvcnRpbmcgdGhlbSBvciB1c2luZyBhIEdQVS4KClRoaXMgY2hlY2tzIGludGVncml0eSBvbmx5LiBJdCBjYW5ub3QgZXN0YWJsaXNoIGFjY291bnQgYXR0YWNobWVudCwgbGljZW5jZQpwZXJtaXNzaW9uLCBydW50aW1lIGNvbXBhdGliaWxpdHkgb3IgY29tcHV0ZSBhdXRob3JpemF0aW9uLgoiIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCByZQppbXBvcnQgdGltZQoKUEFDS0FHRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKUElOID0gcmUuY29tcGlsZShyJyhbQS1aYS16MC05Xy4tXSspPT0oW15cc10rKSg/OiAtLWhhc2g9c2hhMjU2OihbYS1mMC05XXs2NH0pKT8nKQoKCmNsYXNzIFByZWZsaWdodEZhaWxlZChWYWx1ZUVycm9yKToKICAgIHBhc3MKCgpkZWYgbm9ybWFsaXplKG5hbWUpOgogICAgcmV0dXJuIHJlLnN1YihyJ1stXy5dKycsICctJywgbmFtZSkubG93ZXIoKQoKCmRlZiBwaW5zKHRleHQsIHJlcXVpcmVfaGFzaGVzKToKICAgIHJvd3MgPSB7fQogICAgZm9yIGxpbmUgaW4gdGV4dC5zcGxpdGxpbmVzKCk6CiAgICAgICAgaWYgbm90IGxpbmUuc3RyaXAoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBtYXRjaCA9IFBJTi5mdWxsbWF0Y2gobGluZSkKICAgICAgICBpZiBub3QgbWF0Y2ggb3IgKHJlcXVpcmVfaGFzaGVzIGFuZCBub3QgbWF0Y2hbM10pOgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ3JlcXVpcmVtZW50cyBsaW5lIGlzIG5vdCBhbiBleGFjdCBwZXJtaXR0ZWQgcGluJykKICAgICAgICBuYW1lLCB2ZXJzaW9uLCBkaWdlc3QgPSBtYXRjaC5ncm91cHMoKQogICAgICAgIG5hbWUgPSBub3JtYWxpemUobmFtZSkKICAgICAgICBpZiBuYW1lIGluIHJvd3M6CiAgICAgICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgnZHVwbGljYXRlIHJlcXVpcmVtZW50OiAnICsgbmFtZSkKICAgICAgICByb3dzW25hbWVdID0gKHZlcnNpb24sIGRpZ2VzdCkKICAgIHJldHVybiByb3dzCgoKZGVmIGRpZ2VzdF9maWxlKHBhdGgsIGNoZWNrKToKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICBjaGVjaygpCiAgICAgICAgICAgIGNodW5rID0gc3RyZWFtLnJlYWQoOCAqIDEwMjQgKiAxMDI0KQogICAgICAgICAgICBjaGVjaygpCiAgICAgICAgICAgIGlmIG5vdCBjaHVuazoKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIGxvYWRfaW5wdXRzKHBhY2thZ2U9UEFDS0FHRSk6CiAgICBwYWNrYWdlID0gUGF0aChwYWNrYWdlKQogICAgcHJvcG9zYWwgPSBqc29uLmxvYWRzKChwYWNrYWdlIC8gJ3Byb3Bvc2FsLmpzb24nKS5yZWFkX2J5dGVzKCkpCiAgICBtYW5pZmVzdCA9IGpzb24ubG9hZHMoKHBhY2thZ2UgLyAndHJ1c3RlZF9tYW5pZmVzdC5qc29uJykucmVhZF9ieXRlcygpKQogICAgbG9jayA9IChwYWNrYWdlIC8gJ3RydXN0ZWRfcmVxdWlyZW1lbnRzLmxvY2snKS5yZWFkX2J5dGVzKCkKICAgIGNhbm9uaWNhbCA9IGpzb24uZHVtcHMobWFuaWZlc3RbJ2FydGlmYWN0cyddLCBzb3J0X2tleXM9VHJ1ZSwgc2VwYXJhdG9ycz0oJywnLCAnOicpKS5lbmNvZGUoKQogICAgaWYgaGFzaGxpYi5zaGEyNTYoY2Fub25pY2FsKS5oZXhkaWdlc3QoKSAhPSBwcm9wb3NhbFsndHJ1c3RlZF9hcnRpZmFjdHNfc2hhMjU2J106CiAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd0cnVzdGVkIGFydGlmYWN0IGludmVudG9yeSBkaWZmZXJzIGZyb20gaXRzIGJpbmRpbmcnKQogICAgaWYgaGFzaGxpYi5zaGEyNTYobG9jaykuaGV4ZGlnZXN0KCkgIT0gcHJvcG9zYWxbJ3RydXN0ZWRfcmVxdWlyZW1lbnRzX3NoYTI1NiddOgogICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgndHJ1c3RlZCBoYXNoLXBpbm5lZCByZXF1aXJlbWVudHMgZGlmZmVyIGZyb20gdGhlaXIgYmluZGluZycpCiAgICByb3dzID0gbWFuaWZlc3RbJ2FydGlmYWN0cyddCiAgICBuYW1lcyA9IFtyb3dbJ2ZpbGVuYW1lJ10gZm9yIHJvdyBpbiByb3dzXQogICAgaWYgbGVuKHJvd3MpICE9IHByb3Bvc2FsWyd3aGVlbF9jb3VudCddIG9yIGxlbihzZXQobmFtZXMpKSAhPSBsZW4ocm93cyk6CiAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd3cm9uZyB0cnVzdGVkIHdoZWVsIGNvdW50IG9yIGR1cGxpY2F0ZSBmaWxlbmFtZScpCiAgICBmb3IgbmFtZSBpbiBuYW1lczoKICAgICAgICBpZiBub3QgbmFtZS5lbmRzd2l0aCgnLndobCcpIG9yIFBhdGgobmFtZSkubmFtZSAhPSBuYW1lIG9yICcvJyBpbiBuYW1lIG9yICdcXCcgaW4gbmFtZToKICAgICAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd1bnNhZmUgdHJ1c3RlZCBmaWxlbmFtZScpCiAgICBsb2NrX3BpbnMgPSBwaW5zKGxvY2suZGVjb2RlKCd1dGYtOCcpLCByZXF1aXJlX2hhc2hlcz1UcnVlKQogICAgZXhwZWN0ZWRfcGlucyA9IHt9CiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgbmFtZSwgdmVyc2lvbiA9IHJvd1snZmlsZW5hbWUnXS5zcGxpdCgnLScpWzoyXQogICAgICAgIG5hbWUgPSBub3JtYWxpemUobmFtZSkKICAgICAgICBpZiBuYW1lIGluIGV4cGVjdGVkX3BpbnM6CiAgICAgICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgnbXVsdGlwbGUgdHJ1c3RlZCB3aGVlbHMgZm9yIG9uZSBkaXN0cmlidXRpb24nKQogICAgICAgIGV4cGVjdGVkX3BpbnNbbmFtZV0gPSAodmVyc2lvbiwgcm93WydzaGEyNTYnXSkKICAgIGlmIGxvY2tfcGlucyAhPSBleHBlY3RlZF9waW5zOgogICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgndHJ1c3RlZCBsb2NrIGFuZCB3aGVlbCBpbnZlbnRvcnkgZGlzYWdyZWUnKQogICAgcmV0dXJuIHByb3Bvc2FsLCByb3dzLCBsb2NrX3BpbnMKCgpkZWYgdmVyaWZ5X21vdW50ZWQoZGF0YXNldF9yb290LCAqLCBwYWNrYWdlPVBBQ0tBR0UsIG5vdz10aW1lLm1vbm90b25pYyk6CiAgICBzdGFydGVkID0gbm93KCkKICAgIHByb3Bvc2FsLCBhcnRpZmFjdHMsIHRydXN0ZWRfcGlucyA9IGxvYWRfaW5wdXRzKHBhY2thZ2UpCiAgICBkZWFkbGluZSA9IHN0YXJ0ZWQgKyBwcm9wb3NhbFsnaW50ZWdyaXR5X2J1ZGdldF9zZWNvbmRzJ10KCiAgICBkZWYgY2hlY2soKToKICAgICAgICBjdXJyZW50ID0gbm93KCkKICAgICAgICBpZiBjdXJyZW50ID49IGRlYWRsaW5lOgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ2ludGVncml0eSBkZWFkbGluZSBleGNlZWRlZCcpCiAgICAgICAgcmV0dXJuIGN1cnJlbnQKCiAgICByb290ID0gUGF0aChkYXRhc2V0X3Jvb3QpCiAgICBpZiBub3Qgcm9vdC5pc19kaXIoKToKICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ2RhdGFzZXQgcm9vdCBpcyBub3QgbW91bnRlZCcpCiAgICBleHBlY3RlZCA9IHtyb3dbJ2ZpbGVuYW1lJ10gZm9yIHJvdyBpbiBhcnRpZmFjdHN9IHwgc2V0KHByb3Bvc2FsWydwdWJsaXNoZXJfbWV0YWRhdGFfc2hhMjU2J10pCiAgICBwcmVzZW50ID0gc2V0KCkKICAgIGZvciBwYXRoIGluIHJvb3Qucmdsb2IoJyonKToKICAgICAgICBjaGVjaygpCiAgICAgICAgaWYgcGF0aC5pc19zeW1saW5rKCk6CiAgICAgICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgnc3ltbGluayBpbiBkYXRhc2V0IGludmVudG9yeScpCiAgICAgICAgaWYgcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIHByZXNlbnQuYWRkKHBhdGgucmVsYXRpdmVfdG8ocm9vdCkuYXNfcG9zaXgoKSkKICAgIGlmIHByZXNlbnQgIT0gZXhwZWN0ZWQ6CiAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCdkYXRhc2V0IGZpbGUgaW52ZW50b3J5IGRpZmZlcnM6IG1pc3Npbmc9JXMgZXh0cmE9JXMnICUKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKHNvcnRlZChleHBlY3RlZCAtIHByZXNlbnQpWzo1XSwgc29ydGVkKHByZXNlbnQgLSBleHBlY3RlZClbOjVdKSkKICAgIGZvciBuYW1lLCBkaWdlc3QgaW4gcHJvcG9zYWxbJ3B1Ymxpc2hlcl9tZXRhZGF0YV9zaGEyNTYnXS5pdGVtcygpOgogICAgICAgIGlmIGRpZ2VzdF9maWxlKHJvb3QgLyBuYW1lLCBjaGVjaykgIT0gZGlnZXN0OgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ3B1Ymxpc2hlciBtZXRhZGF0YSBkaWZmZXJzOiAnICsgbmFtZSkKICAgIHB1Ymxpc2hlcl9waW5zID0gcGlucygocm9vdCAvICdyZXF1aXJlbWVudHMubG9jaycpLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnKSwgcmVxdWlyZV9oYXNoZXM9RmFsc2UpCiAgICBpZiB7bjogdiBmb3IgbiwgKHYsIF8pIGluIHB1Ymxpc2hlcl9waW5zLml0ZW1zKCl9ICE9IHtuOiB2IGZvciBuLCAodiwgXykgaW4gdHJ1c3RlZF9waW5zLml0ZW1zKCl9OgogICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgncHVibGlzaGVyIHBhY2thZ2UvdmVyc2lvbiBwaW5zIGRpZmZlcicpCiAgICBmb3Igcm93IGluIGFydGlmYWN0czoKICAgICAgICBjaGVjaygpCiAgICAgICAgcGF0aCA9IHJvb3QgLyByb3dbJ2ZpbGVuYW1lJ10KICAgICAgICBpZiBwYXRoLnN0YXQoKS5zdF9zaXplICE9IHJvd1snc2l6ZSddOgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ3doZWVsIHNpemUgZGlmZmVyczogJyArIHJvd1snZmlsZW5hbWUnXSkKICAgICAgICBpZiBkaWdlc3RfZmlsZShwYXRoLCBjaGVjaykgIT0gcm93WydzaGEyNTYnXToKICAgICAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd3aGVlbCBieXRlIGhhc2ggZGlmZmVyczogJyArIHJvd1snZmlsZW5hbWUnXSkKICAgIGZpbmlzaGVkID0gY2hlY2soKSAgIyBleHBsaWNpdCBjaGVjayBhZnRlciBhbGwgcmVhZHM7IGFuIG92ZXJydW4gbmV2ZXIgcGFzc2VzCiAgICByZXR1cm4geydpbnRlZ3JpdHlfcGFzc2VkJzogVHJ1ZSwgJ3doZWVsX2J5dGVzX3ZlcmlmaWVkJzogbGVuKGFydGlmYWN0cyksCiAgICAgICAgICAgICd0cnVzdGVkX2FydGlmYWN0c19zaGEyNTYnOiBwcm9wb3NhbFsndHJ1c3RlZF9hcnRpZmFjdHNfc2hhMjU2J10sCiAgICAgICAgICAgICdpbnN0YWxsYXRpb25fcmVxdWlyZW1lbnRzX3NoYTI1Nic6IHByb3Bvc2FsWyd0cnVzdGVkX3JlcXVpcmVtZW50c19zaGEyNTYnXSwKICAgICAgICAgICAgJ2RhdGFzZXRfcmVmJzogcHJvcG9zYWxbJ2RhdGFzZXQnXVsncmVmJ10sICdyZXF1ZXN0ZWRfdmVyc2lvbic6IHByb3Bvc2FsWydkYXRhc2V0J11bJ3ZlcnNpb24nXSwKICAgICAgICAgICAgJ2FjY291bnRfYXR0YWNobWVudF92ZXJpZmllZCc6IEZhbHNlLCAncHJvdmlkZXJfYXR0YWNobWVudF92ZXJzaW9uX3ZlcmlmaWVkJzogRmFsc2UsCiAgICAgICAgICAgICdwZXJtaXNzaW9uc19hc3Nlc3NlZCc6IEZhbHNlLCAnZ3B1X2NvbXBhdGliaWxpdHlfZXN0YWJsaXNoZWQnOiBGYWxzZSwKICAgICAgICAgICAgJ2xhdW5jaF9hdXRob3JpemVkJzogRmFsc2UsICdlbGFwc2VkX3NlY29uZHMnOiByb3VuZChmaW5pc2hlZCAtIHN0YXJ0ZWQsIDMpfQo=', 'proposal.json': 'ewogICJzY2hlbWEiOiAiZGlyZWN0X3B1Ymxpc2hlcl9zbW9rZV9wcmVwYXJhdGlvbl92MSIsCiAgInNjb3BlIjogImRpcmVjdC1wdWJsaXNoZXItc21va2UtdjEiLAogICJzdGF0dXMiOiAiY3B1X2ludGFrZV9kcmFmdF9ub3RfYV9ncHVfc21va2VfbGF1bmNoX3BhY2thZ2UiLAogICJzb3VyY2VfYmFzaXNfY29tbWl0IjogIjMxOGM1Y2IiLAogICJkYXRhc2V0IjogewogICAgInJlZiI6ICJkcmllc3NtaXQxL2FyYzMtdmxsbS1oMTAwLXdoZWVsaG91c2UtdjMiLAogICAgInZlcnNpb24iOiAxCiAgfSwKICAidHJ1c3RlZF9hcnRpZmFjdHNfc2hhMjU2IjogIjM2OTFjYjg4NTRkZjRkOGZmMTBlNDJjYTk5NTdmZGRiOWE4YWUwMzYyMDY0ZTdiMzFiMzIwNTg5MWFmMGQ1NDYiLAogICJ0cnVzdGVkX3JlcXVpcmVtZW50c19zaGEyNTYiOiAiYmE4MGQzNTA2MjI0NTQyMWRhZjFjYWU2NTQ3NDI4MTk1MmNjMGM0NGZiNDZlMTFjZjdmNjhkMGVjZTQ5NjQwNiIsCiAgIndoZWVsX2NvdW50IjogMTc0LAogICJpbnRlZ3JpdHlfYnVkZ2V0X3NlY29uZHMiOiA5MDAsCiAgInB1Ymxpc2hlcl9tZXRhZGF0YV9zaGEyNTYiOiB7CiAgICAiUkVBRE1FLm1kIjogImQyZThkYTZkMzZjMjQzMTk4ODgyZTZjYjY4ZDYwZDcxMjFkOGFhOTY0MmU1NWQ3ZmNjMzg0YmM0ZDA4N2I2MzEiLAogICAgIlNIQTI1NlNVTVMiOiAiODA1Mzg4ZWZiNmJmN2YwYTI0OGYyYmI3MDc5ODE4ODczNzA3NzU0NTE1OWZlMGU0NzVhZWU0ZDQ0ODQzYzkwMSIsCiAgICAicmVxdWlyZW1lbnRzLmxvY2siOiAiYmIzZTMwYWM1ZTMyNzQ1NmUzOTE1YjgwN2RhOTYwMjAzYzIwYmMxYWEzNTIyMDRlMzlmNjZkNTYxOGE2YmNiMCIKICB9LAogICJ1c2VfZmFjdHMiOiB7CiAgICAic3RhdHVzIjogImRlcGxveW1lbnRfZmFjdHNfcmV0YWluZWRfbG9jYWxseV9ub3RfcHVibGlzaGVkIiwKICAgICJyZXF1aXJlZF9mb3JfcmV2aWV3IjogWwogICAgICAiY29uc3VtaW5nIGFjY291bnQiLAogICAgICAibGljZW5jZSBob2xkZXIgYW5kIHRlYW1tYXRlIHJvbGVzIiwKICAgICAgImFjY2VzcyBjb250cm9scyIsCiAgICAgICJwdWJsaWNhdGlvbiBpbnRlbnQiLAogICAgICAic2VwYXJhdGUgdmVuZG9yIGFncmVlbWVudCwgaWYgYW55IiwKICAgICAgIm91dHB1dCBhbmQgcGF5bG9hZCBoYW5kbGluZyIKICAgIF0sCiAgICAicmV2aWV3X291dGNvbWUiOiAibm90IGFzc2Vzc2VkIG9yIGFwcHJvdmVkIgogIH0sCiAgInVucmVzb2x2ZWRfdXNlX3F1ZXN0aW9ucyI6IFsKICAgICJXaGF0IGxlZ2FsIGVudGl0eSwgaWYgYW55LCBpcyByZXByZXNlbnRlZCBieSB0aGUgYWNjZXB0aW5nIGFjY291bnQgaG9sZGVyPyIsCiAgICAiRG8gY29sbGFib3JhdG9ycyBpbmRlcGVuZGVudGx5IGFjY2VwdCBhcHBsaWNhYmxlIGxpY2VuY2VzLCBvciBxdWFsaWZ5IHVuZGVyIHRoZSBhcHBsaWNhYmxlIGF1dGhvcml6ZWQtdXNlciB0ZXJtcz8iLAogICAgIkRvZXMgdGhlIGhvc3RlZCBhY2Nlc3MgYW5kIGNvcHlpbmcgbW9kZWwgc2F0aXNmeSB0aGUgZXhhY3QgYXBwbGljYWJsZSBsaWNlbmNlIHRlcm1zPyIsCiAgICAiV2hpY2ggaW5zdGFsbGVkIHBheWxvYWRzLCBub3RlYm9vayBvdXRwdXRzLCBhcmNoaXZlcyBvciBpbWFnZXMgYXJlIHJldGFpbmVkL2V4cG9ydGVkL3NoYXJlZD8iLAogICAgIldoaWNoIGV4YWN0IGdvdmVybmluZyB0ZXh0cyBhcHBseSB0byBjdUROTiwgY3VTUEFSU0VMdCwgTlZTSE1FTSBhbmQgQ1VEQSBzdWJjb21wb25lbnRzPyIKICBdLAogICJnYXRlcyI6IHsKICAgICJhY2NvdW50X2F0dGFjaG1lbnQiOiAidW52ZXJpZmllZDsgbm8gY3JlZGVudGlhbHMgYXZhaWxhYmxlIGluIGNoZWNrZWQgbG9jYXRpb25zIiwKICAgICJwcm92aWRlcl9hdHRhY2htZW50X3ZlcnNpb24iOiAidW52ZXJpZmllZCIsCiAgICAibW91bnRlZF93aGVlbF9ieXRlX3ZlcmlmaWNhdGlvbiI6ICJub3QgcnVuIiwKICAgICJkaXJlY3RfY29uc3VtcHRpb25fcGVybWlzc2lvbiI6ICJub3QgYXNzZXNzZWQgb3IgYXBwcm92ZWQiLAogICAgInNlcGFyYXRlX3Ntb2tlX3J1bm5lciI6ICJub3QgaW1wbGVtZW50ZWQ7IHRoaXMgcGFja2FnZSBpcyBDUFUgaW50YWtlIG9ubHkiLAogICAgImZpbmFsX3Ntb2tlX3BhY2thZ2VfcmV2aWV3IjogIm5vdCBwZXJmb3JtZWQiLAogICAgImNvbXB1dGVfYXV0aG9yaXphdGlvbiI6ICJhYnNlbnQiCiAgfSwKICAicHJvcG9zZWRfc21va2UiOiB7CiAgICAic3RhdHVzIjogInByb3Bvc2FsX29ubHlfbm90X2F1dGhvcml6ZWQiLAogICAgInByaW9yX2NvbnRyb2xfaGFybmVzcyI6IHsKICAgICAgImJyYW5jaCI6ICJ3aGVlbGhvdXNlLXIyLXNtb2tlLXYxIiwKICAgICAgImNvbW1pdCI6ICJiMDdlMmIyIgogICAgfSwKICAgICJtb2RlbF9pZCI6ICJRd2VuL1F3ZW4zLVZMLTMwQi1BM0ItSW5zdHJ1Y3QtRlA4IiwKICAgICJtb2RlbF9yZXZpc2lvbiI6ICJkOTc0OGE1MWFlNjYzNTRjNGRhZDY2NWFhYjJjNzFmMjZjZjJjOGNkIiwKICAgICJtb2RlbF90cmVlX3NoYTI1NiI6ICIwNTJhYjI3ZjA2YzI4MjYxZTE0M2I4YzE2MzgzODJkMTA3YjAzNDY5MmJjMGNkMTc5MmVjNGUwMmRkYWI4NjI3IiwKICAgICJydW50aW1lIjogewogICAgICAicHl0aG9uIjogIjMuMTIiLAogICAgICAiZ2xpYmNfbWluaW11bSI6ICIyLjM0IiwKICAgICAgIm1hY2hpbmUiOiAieDg2XzY0IiwKICAgICAgImdwdV9jb3VudCI6IDEsCiAgICAgICJncHVfbmFtZV9jb250YWlucyI6ICJSVFggUFJPIDYwMDAiLAogICAgICAicGFja2FnZXMiOiB7CiAgICAgICAgInRvcmNoIjogIjIuMTAuMCtjdTEyOCIsCiAgICAgICAgInZsbG0iOiAiMC4xOS4wIiwKICAgICAgICAidHJhbnNmb3JtZXJzIjogIjQuNTcuNiIsCiAgICAgICAgIm51bXB5IjogIjIuMi42IgogICAgICB9CiAgICB9LAogICAgImxpbWl0cyI6IHsKICAgICAgImF1dGhvcml6ZWRfc2Vjb25kcyI6IDM2MDAsCiAgICAgICJpbnRlcm5hbF9zZWNvbmRzIjogMzQyMCwKICAgICAgImNsZWFudXBfcmVzZXJ2ZV9zZWNvbmRzIjogMzAwLAogICAgICAiYWRtaXNzaW9uX2N1dG9mZl9zZWNvbmRzIjogMzEyMCwKICAgICAgImluc3RhbGxhdGlvbl9zZWNvbmRzIjogOTAwLAogICAgICAibWF4aW11bV9hdHRlbXB0cyI6IDEsCiAgICAgICJtYXhpbXVtX21vZGVsX3JlcXVlc3RzIjogMTIsCiAgICAgICJhdXRvbWF0aWNfcmV0cmllcyI6IDAKICAgIH0sCiAgICAiaW5zdGFsbGF0aW9uIjogImZyZXNoIHZlbnY7IG5vIGluZGV4L2NhY2hlOyBiaW5hcnkgb25seTsgdHJ1c3RlZF9yZXF1aXJlbWVudHMubG9jayB3aXRoIC0tcmVxdWlyZS1oYXNoZXM7IHBpcCBjaGVjazsgZXhhY3QgaW1wb3J0cy92ZXJzaW9ucyIsCiAgICAibGlmZWN5Y2xlX3JlcXVpcmVtZW50cyI6ICJwcmVzZXJ2ZSBwcm90ZWN0ZWQgc3RhcnR1cCBvd25lcnNoaXAgYW5kIHVuY2VydGFpbi1vd25lcnNoaXAgY2xlYW51cCwgR1BVIGNsZWFudXAsIGV2aWRlbmNlIGZpbmFsaXphdGlvbiwgdGVtcG9yYXJ5LWVudmlyb25tZW50IHJlbW92YWwgYW5kIGFuIGV4cGxpY2l0IGZpbmFsIGRlYWRsaW5lIGNoZWNrOyBvdmVycnVucyBhbHdheXMgZmFpbCIKICB9Cn0K', 'trusted_manifest.json': '...', 'trusted_requirements.lock': '...'}
bindings = {'__init__.py': '08c32bf5bc6167ad1202120f90a00043271310c34896d33188c78e2049eb2665', 'preflight.py': '369a39a09c4f5f97b0d92613c3ecc6aa281779af4721a4154fe98576e4fc3b0d', 'proposal.json': 'cb5e1cee9903861e16960664c4d392821fb880e4125c7bce1865ea69573a600c', 'trusted_manifest.json': '91ad9ede70c4fdf027cf5739a828f9063dc07c22c6010a64ebd1af0db8f0462b', 'trusted_requirements.lock': 'ba80d35062245421daf1cae65474281952cc0c44fb46e11cf7f68d0ece496406'}
source = pathlib.Path(tempfile.mkdtemp(prefix='direct-publisher-preflight-'))
try:
    package = source / 'certification/direct_publisher_smoke_v1'
    package.mkdir(parents=True)
    for name, value in payload.items():
        data = base64.b64decode(value)
        if hashlib.sha256(data).hexdigest() != bindings[name]:
            raise ValueError('embedded source drift: ' + name)
        (package / name).write_bytes(data)
    sys.path.insert(0, str(source))
    from certification.direct_publisher_smoke_v1.preflight import verify_mounted
    roots = [pathlib.Path('/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3'),
             pathlib.Path('/kaggle/input/arc3-vllm-h100-wheelhouse-v3')]
    mounted = [p for p in roots if p.is_dir()]
    if len(mounted) != 1:
        raise ValueError('expected exactly one known dataset mount; do not guess another source')
    receipt = verify_mounted(mounted[0])
    print(json.dumps(receipt, indent=2))
    output = pathlib.Path('/kaggle/working/direct-publisher-byte-integrity.json')
    output.write_text(json.dumps(receipt, indent=2) + '\n')
finally:
    sys.path.remove(str(source)) if str(source) in sys.path else None
    shutil.rmtree(source)
